# Lab 12 · Delegate an architecture review with A2A

**Prepared by Mustafa Assaf** · Self-study: Build your council further · 15–20 minute lab

## Talk to another agent with A2A

**So far:** in Lab 04, the model called *tools*: functions inside our own program.

**New:** *agents in other programs*. An agent can be written by another team, in another language, on another machine. The **Agent2Agent (A2A)** protocol lets our code find such an agent and talk to it, without importing its code:

1. The agent publishes an *Agent Card*: its name, description, and skills.
2. A client *discovers* the agent by fetching that card.
3. The client sends a *message*, and the agent replies with a message.

We start with a tiny hotel-desk agent, then let a reviewer agent check our model's architecture proposals. The course helper `talk_over_a2a` serves any Python function as an A2A agent and performs all three steps with the official SDK, inside this notebook; `review_over_a2a` uses it for the reviewer. You can read both in [`coursekit/protocol_examples.py`](https://github.com/mkassaf/agentic-ai-course/blob/main/coursekit/protocol_examples.py).

## Pattern · Generator–Critic Loop

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Plans, code, and writing improve through review against explicit criteria rather than one-shot generation. |
| **Problem** | A generator cannot reliably notice its own omissions, and a critique without revision improves nothing. |
| **Solution** | Give generation and review to separate agents, and feed the critic's must-fix feedback into the next draft until a pass condition or an iteration cap. |
| **Use when** | Quality can be checked; revision is cheaper than failure; clear stopping criteria exist. |
| **Trade-off** | Produces stronger results, but a critic can cause endless churn without a strict budget and pass condition. |
| **Flow** | Generate → Critique → Revise → Pass gate |
| **Quality attributes** | Benefits: Accuracy, Reliability, Observability, Modifiability. Liabilities: Latency, Cost, Maintainability, Accuracy, Predictability. |
| **In this lab** | After the tiny hotel-desk agent of Steps 1 and 2, our model generates proposals, and a separate reviewer agent, reached over A2A, is the critic. Its feedback drives one revision; one revision is the iteration cap. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'a2a']):
    %pip install -q "agentic-ai-coursekit[a2a] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · A tiny agent

Before any protocol, an agent here is just a function: text in, text out. The hotel desk reads a city name and answers with the hotels that have free rooms there.

In [ ]:
import json

rooms = {"Lisbon": {"Alfama Inn": 3, "Harbor View": 0}, "Amsterdam": {"Canal House": 2}}

def hotel_desk(text):
    """A tiny agent: it reads a city and answers with the hotels that have free rooms."""
    city = text.strip()
    free = {hotel: n for hotel, n in rooms.get(city, {}).items() if n > 0}
    return f"Free rooms in {city}: {free}" if free else f"No free rooms in {city}."

print(hotel_desk("Lisbon"))

## Step 2 · The same agent, over A2A

`talk_over_a2a` serves `hotel_desk` as an A2A agent, fetches its Agent Card as a client would, and sends one message. The client never imports `hotel_desk`: it learns what the agent offers from the card, and talks to it only through messages.

In [ ]:
from coursekit.protocol_examples import review_over_a2a, talk_over_a2a

DESK = dict(name="Hotel desk", description="Answers which hotels in a city have free rooms.",
            skill="Availability")
card, reply = await talk_over_a2a(hotel_desk, "Lisbon", **DESK)
print("Discovered card:", json.dumps(card, indent=2))
print("Reply:", reply)
_, reply = await talk_over_a2a(hotel_desk, "Paris", **DESK)
print("Reply:", reply)

## Step 3 · A proposer that can revise

`propose` asks our model for a decision. On a retry, it adds the reviewer's feedback to the prompt. The prompt does not contain the policy, so the model can make the same mistakes as in Lab 00. Here the reviewer catches them.

In practice mode, `scripted` holds the replies for each project: the first attempt, then the revision if there is one.

In [ ]:
projects = [
    {"name": "Course portal",     "team_size": 3,  "independent_deploys": False},
    {"name": "Payments platform", "team_size": 12, "independent_deploys": True},
    {"name": "Internal wiki",     "team_size": 9,  "independent_deploys": False},
    {"name": "Mobile backend",    "team_size": 5,  "independent_deploys": True},
]
scripted = {"Course portal": ["modular monolith"], "Payments platform": ["microservices"],
            "Internal wiki": ["microservices", "modular monolith"], "Mobile backend": ["modular monolith"]}

def propose(project, feedback=""):
    """Ask the model for a decision. On a retry, include the reviewer's feedback."""
    prompt = f"Pick modular monolith or microservices for {project}. Reply with the label only."
    if feedback:
        prompt += f"\nA reviewer rejected your last answer: {feedback} Answer again."
    attempt = 1 if feedback else 0
    reply = ask(prompt, example=scripted[project["name"]][attempt])
    return reply.strip().rstrip(".").lower()  # "Modular monolith." -> "modular monolith"

## Step 4 · Propose, review, and revise

Now a reviewer agent checks architecture proposals: `review_over_a2a` sends the project and the proposal as an A2A message, and the reviewer replies `PASS` or `FAIL` with a reason. For each project, our model proposes and the reviewer checks. A rejected proposal gets exactly one revision, so the exchange always ends. Every project should pass, and `revisions` counts the corrections.

In [ ]:
passed, revisions = {}, 0
for project in projects:
    proposal = propose(project)
    _, report = await review_over_a2a(project, proposal)
    print(f"{project['name']:<18} proposed {proposal!r:<19} -> {report}")
    if report.startswith("FAIL"):
        revisions += 1
        proposal = propose(project, feedback=report)
        _, report = await review_over_a2a(project, proposal)
        print(f"{'':<18} revised  {proposal!r:<19} -> {report}")
    passed[project["name"]] = report.startswith("PASS")
print(passed, "| revisions needed:", revisions)
assert all(passed.values()), "A proposal still fails after one revision."

## What you should see

Step 1 prints `Free rooms in Lisbon: {'Alfama Inn': 3}`. Step 2 prints the discovered Agent Card, with its name, description, skill, and address, then the same answer, now received over A2A, and `No free rooms in Paris.` In Step 4, three projects pass at once. The Internal wiki is first proposed as `'microservices'` and fails, then the revision `'modular monolith'` passes. The last line shows every project as `True` and `revisions needed: 1`.

## Try it

1. Ask the hotel desk about Amsterdam. Then change its card's `description`. What does a client learn from the card, and what can it not learn?
2. Write a second tiny agent, for example a currency desk that converts euros to dollars at a fixed rate, and call it with `talk_over_a2a`.
3. Make the Mobile backend's first scripted reply `"microservices"`, and add `"modular monolith"` as its revision. Run Steps 3 and 4. Now two projects need a revision.
4. Make a revision wrong too, for example `["microservices", "microservices"]`. The final check stops the run.
5. In live mode, the model's revisions are real. Does a real model take the reviewer's feedback?
6. **Use it in your council.** Run one architect from Lab 06 as an A2A agent with its own Agent Card. What must each message carry so that the chair's transcript stays traceable?

## Takeaway

A tool is a function inside our program; an A2A agent is a program of its own. A2A lets one agent find another through its Agent Card and exchange messages with it, without sharing code. A separate reviewer gives a second, independent check, and its feedback can drive a revision. Lab 13 builds the same choose, check, and retry loop with a framework.

## Read further

- [Official A2A SDK](https://github.com/a2aproject/a2a-python)
- [Agent discovery](https://a2a-protocol.org/latest/topics/agent-discovery/)